# docextract on Colab: baselines, fine-tuning, results on every document type
Runtime > Change runtime type > **GPU**. Everything is driven by `configs/*.yaml`; results and checkpoints go to Google Drive so a disconnect loses nothing (runs resume).

Order: setup > data > baselines (all engines x all types) > smoke-train > train > evaluate the fine-tuned model > save.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/document-capture-runs'
!mkdir -p {DRIVE}
!rm -rf /content/document-capture && git clone -q https://github.com/aamna2401/document-capture.git /content/document-capture
%cd /content/document-capture
!git log --oneline | head -3

In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip -q install -r requirements-colab.txt

## 1. Data (public: FUNSD forms, arXiv scanned pages, CORD receipts)

In [ ]:
import os
os.makedirs('data/public/arxiv', exist_ok=True)
if not os.path.exists('data/public/funsd_raw/dataset'):
    !curl -sL -o data/public/funsd.zip https://guillaumejaume.github.io/FUNSD/dataset.zip
    !unzip -q -o data/public/funsd.zip -d data/public/funsd_raw
for pid in "1706.03762 1810.04805 1512.03385 1409.1556 1412.6980 1502.03167 2010.11929 1706.02677".split():
    if not os.path.exists(f'data/public/arxiv/{pid}.pdf'):
        !curl -sL -o data/public/arxiv/{pid}.pdf https://arxiv.org/pdf/{pid}
        !sleep 3

from datasets import load_dataset          # CORD-v2, CC BY 4.0
for split in ['train', 'test']:
    if not os.path.exists(f'data/cord_{split}.parquet'):
        load_dataset('naver-clova-ix/cord-v2', split=split).to_parquet(f'data/cord_{split}.parquet')
!python scripts/make_gold.py

## 2. Baselines: every engine on every document type

In [ ]:
!python scripts/run_all.py configs/experiment.yaml --out {DRIVE}/runs --run-id baseline

## 3. Train the recognizer\nTraining crops come from FUNSD *train* and CORD *train* only; the split is by form, and a test asserts no form is in both train and validation. Test sets are never read here.

In [ ]:
!python scripts/prepare_crops.py --cord-train data/cord_train.parquet

In [ ]:
# Smoke run first: 20 steps, proves the whole loop works before spending GPU time
import yaml
cfg = yaml.safe_load(open('configs/train_recognizer.yaml'))
cfg.update(smoke=True, out_dir='outputs/recognizer_smoke'); yaml.safe_dump(cfg, open('/content/smoke.yaml', 'w'))
!python scripts/train_recognizer.py /content/smoke.yaml

In [ ]:
cfg = yaml.safe_load(open('configs/train_recognizer.yaml'))
cfg.update(out_dir=f'{DRIVE}/recognizer'); yaml.safe_dump(cfg, open('/content/train.yaml', 'w'))
!python scripts/train_recognizer.py /content/train.yaml

## 4. Evaluate the fine-tuned model on the untouched test sets

In [ ]:
ex = yaml.safe_load(open('configs/experiment.yaml'))
ex['engines'] = {'rapidocr': {'name': 'rapidocr'},
                 'trocr_pretrained': ex['engines']['trocr_pretrained'],
                 'trocr_finetuned': {'name': 'trocr', 'options': {'checkpoint': f'{DRIVE}/recognizer/best', 'label': 'trocr_finetuned'}}}
ex['run_name'] = 'finetuned'; yaml.safe_dump(ex, open('/content/exp_ft.yaml', 'w'))
!python scripts/run_all.py /content/exp_ft.yaml --out {DRIVE}/runs --run-id finetuned

## 5. Results

In [ ]:
import pandas as pd
pd.set_option('display.width', 200)
for run in ['baseline', 'finetuned']:
    print(run); display(pd.read_csv(f'{DRIVE}/runs/{run}/summary.csv'))

Read the summary honestly: `f1_ci_lo/hi` is a 95% bootstrap interval over documents, `share_f1>=0.9` is the fraction of documents at the target. Compare `trocr_finetuned` with `trocr_pretrained` (did training help?) and with `rapidocr` (is it worth it?). Per-file CSVs and `meta.json` (git commit, versions, GPU, seed) sit next to each summary.